## 🔒 API Key Setup Instructions

Before running this notebook, please follow these steps to securely provide your API key.

1. In the same folder as this notebook, **create a new file** named `.env`
2. Add the following line inside it: OPENAI_API_KEY=your_api_key_here

3. You can create a **free API key** from here:  
👉 [OpenRouter – Alibaba Tongyi DeepResearch 30B (Free)](https://openrouter.ai/alibaba/tongyi-deepresearch-30b-a3b:free)

⚠️ **Never upload your API key** to GitHub, share it publicly, or write it directly in the code.


In [ ]:
"alibaba/tongyi-deepresearch-30b-a3b:free"

In [2]:
#link of  the p/content/Nour_Ardhaoui (5).pdfdf
DATA_PATH="/content/resume_juanjosecarin.pdf"
model_name="openai/gpt-oss-120b:free"


In [3]:
#link to local chroma
CHROMA_PATH="chroma"

In [35]:
api_key2="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"
api_key="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"
api_key3="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"
API_URL = "https://resumeparser-api.p.rapidapi.com/"
API_KEY = "e65ea1c6e8msh51cc8c8fbe5a166p17b753jsn73fead402040"
API_HOST = "resumeparser-api.p.rapidapi.com"

# load and install libraries

In [ ]:
# TO DO:
# when the llm returns {} i try to return failed json
# add another api and another model


In [5]:
from dotenv import load_dotenv
load_dotenv()

False

In [6]:
import os

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✅ API key loaded successfully!")
else:
    print("⚠️ API key not found. Check your .env file.")


⚠️ API key not found. Check your .env file.


In [7]:
!pip install -U langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.0 MB/s eta 0:00:00


In [8]:
!pip install langchain
!pip install langchain-community
!pip install chromadb
!pip install sentence-transformers
!pip install pypdf
!pip install transformers
!pip install torch
!pip install -U langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [9]:
!pip install chromadb sentence-transformers pypdf transformers torch

In [10]:
!pip install json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.3 MB/s eta 0:00:00


In [11]:
!pip install langchain --upgrade
!pip install langchain-core --upgrade
!pip install langchain-community --upgrade
!pip install langchain-huggingface --upgrade

In [12]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

In [13]:
import requests
import logging

In [14]:
from langchain_community.document_loaders import PyPDFLoader

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
from langchain_community.vectorstores import Chroma


In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

In [18]:
from langchain_openai import ChatOpenAI

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [20]:
from uuid import uuid4
from json_repair import repair_json
import json
import re

# load embedding model and chromadb

### embedding model

In [21]:
"""
| Model                                     | Type                  | Notes                                        |
| ----------------------------------------- | --------------------- | -------------------------------------------- |
| `all-MiniLM-L6-v2` (SentenceTransformers) | Small, fast           | Great for clustering and search, lightweight |
| `all-mpnet-base-v2`                       | Medium, more accurate | Excellent for semantic similarity            |
| `sentence-transformers/LaBSE`             | Multilingual          | If you handle resumes in multiple languages  |
| `intfloat/e5-large`                       | High-quality          | Very good for semantic search, longer texts  |


"""

'\n| Model                                     | Type                  | Notes                                        |\n| ----------------------------------------- | --------------------- | -------------------------------------------- |\n| `all-MiniLM-L6-v2` (SentenceTransformers) | Small, fast           | Great for clustering and search, lightweight |\n| `all-mpnet-base-v2`                       | Medium, more accurate | Excellent for semantic similarity            |\n| `sentence-transformers/LaBSE`             | Multilingual          | If you handle resumes in multiple languages  |\n| `intfloat/e5-large`                       | High-quality          | Very good for semantic search, longer texts  |\n\n\n'

In [22]:
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [23]:
embeddings_model

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### vectore database

In [24]:
vector_store=Chroma(
    collection_name="cv_collection",
    embedding_function=embeddings_model,
    persist_directory=CHROMA_PATH,
)

/tmp/ipykernel_1747/2588183526.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store=Chroma(


In [25]:
loader = PyPDFLoader(DATA_PATH)
raw_documents = loader.load()

print(f"Loaded {len(raw_documents)} pages")
print(raw_documents[0].page_content[:500])  # show first 500 chars of first page


Loaded 2 pages
1 of 2 
Juan Jose Carin 
Data Scientist 
 
Mountain View, CA 94041 
 650-336-4590 | juanjose.carin@gmail.com 
 linkedin.com/in/juanjosecarin | juanjocarin.github.io 
 
Professional Profile 
Passionate abo ut data analysis and experiments, mainly focused on user behavior, experience, and engagement , with a solid 
background in data science and statistics, and extensive experience using data insights to drive business growth.  
Education
2016 University of California, Berkeley Master of Informati


In [26]:
raw_documents

[Document(metadata={'producer': 'ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2016-11-08T11:30:19-08:00', 'moddate': '2016-11-08T20:35:18+01:00', 'title': 'Resume - Juan Jose Carin', 'author': 'Juan Jose Carin', 'source': '/content/resume_juanjosecarin.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content="1 of 2 \nJuan Jose Carin \nData Scientist \n \nMountain View, CA 94041 \n 650-336-4590 | juanjose.carin@gmail.com \n linkedin.com/in/juanjosecarin | juanjocarin.github.io \n \nProfessional Profile \nPassionate abo ut data analysis and experiments, mainly focused on user behavior, experience, and engagement , with a solid \nbackground in data science and statistics, and extensive experience using data insights to drive business growth.  \nEducation\n2016 University of California, Berkeley Master of Information and Data Science GPA: 3.93\n \n \n \nRelevant courses: \n• Machine Learning \n• Machine Learning at Scale \n• Storing and Retrieving Data \n•

In [27]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(raw_documents)
print(f"Total chunks: {len(chunks)}")

Total chunks: 8


In [28]:
chunks

[Document(metadata={'producer': 'ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2016-11-08T11:30:19-08:00', 'moddate': '2016-11-08T20:35:18+01:00', 'title': 'Resume - Juan Jose Carin', 'author': 'Juan Jose Carin', 'source': '/content/resume_juanjosecarin.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='1 of 2 \nJuan Jose Carin \nData Scientist \n \nMountain View, CA 94041 \n 650-336-4590 | juanjose.carin@gmail.com \n linkedin.com/in/juanjosecarin | juanjocarin.github.io \n \nProfessional Profile \nPassionate abo ut data analysis and experiments, mainly focused on user behavior, experience, and engagement , with a solid \nbackground in data science and statistics, and extensive experience using data insights to drive business growth.  \nEducation\n2016 University of California, Berkeley Master of Information and Data Science GPA: 3.93\n \n \n \nRelevant courses: \n• Machine Learning \n• Machine Learning at Scale \n• Storing and Retrieving Data \n•

In [ ]:
uuids = [str(uuid4()) for _ in range(len(chunks))]

# adding chunks to vector store
vector_store.add_documents(documents=chunks, ids=uuids)

['969cdebc-fca0-471a-bb2b-42800680831e',
 '08fbd163-ab1f-4466-b89d-34c3107a8525',
 'be6d2a1c-2607-4a61-87aa-89828cc6315e',
 'e881ef5b-57e3-4092-bb0d-c46a52dc6425',
 '1701934c-db4d-42e1-86dc-a6b004b8ae08',
 '93dff653-e934-4f02-b8a6-640cb9e3944b',
 '1991becd-64c0-4b6a-8075-299a2bd5c1e8',
 'f53c6c8f-0f28-4a9c-b7b4-e66460529741']

In [30]:
vector_store.persist()

/tmp/ipykernel_1747/485603143.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


# set the RAG

### set the retriever and llms

In [31]:
def parse_with_rapidapi(data_path, api_key, api_host, api_url, timeout=30):
    """Parse a resume using RapidAPI."""
    headers = {
        "x-rapidapi-key": api_key,
        "x-rapidapi-host": api_host
    }

    try:
        with open(data_path, "rb") as f:
            files = {"resume": (data_path, f, "application/pdf")}
            response = requests.post(api_url, headers=headers, files=files, timeout=timeout)

        response.raise_for_status()

        try:
            data = response.json()
            logger.info("RapidAPI parsing succeeded.")
            return data
        except json.JSONDecodeError:
            logger.error("❌ Failed to parse JSON response from RapidAPI.")
            return None

    except FileNotFoundError:
        logger.error(f"❌ File not found: {data_path}")
        return None
    except requests.exceptions.Timeout:
        logger.error("⏰ RapidAPI request timed out.")
        return None
    except requests.exceptions.RequestException as e:
        logger.error(f"❌ RapidAPI request failed: {e}")
        return None
    except Exception as e:
        logger.error(f"⚠️ Unexpected error with RapidAPI: {e}")
        return None



In [ ]:
"""def setup_llm(api_key,model_name):
  llm=ChatOpenAI(
    model_name=model_name,
    openai_api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    model_kwargs={"response_format": {"type": "json_object"}},
    max_retries=3,
    request_timeout=60, )
  prompt = ChatPromptTemplate.from_template(
    """Answer the question based on the following context:
       {context}

     Question: {question}

       Answer:"""
    )
  retriever = vector_store.as_retriever(search_kwargs={"k": 7})
  qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
  )
  return qa_chain"""

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 14)

In [32]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def setup_llm(api_key, model_name):

    llm = ChatOpenAI(
        model=model_name,
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
        temperature=0,
        max_retries=3,
        request_timeout=60
    )

    prompt = ChatPromptTemplate.from_template("""
    Answer the question based on the following context:

    Context:
    {context}

    Question:
    {question}

    Answer:
    """)

    retriever = vector_store.as_retriever(search_kwargs={"k": 7})

    qa_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return qa_chain

In [36]:
llm_model = setup_llm(api_key,model_name)
llm_model2=setup_llm(api_key2,model_name)
llm_model3=setup_llm(api_key3,model_name)

### set the queries

In [37]:
def clean_text(ch):
  ch = ch.replace('json', "")
  ch = ch.replace('```', "")
  ch = re.sub(r'\s+', ' ', ch)
  ch = ch.replace("None", "null")
  ch = ch.encode('ascii', 'ignore').decode('ascii')
  return ch.strip()


In [38]:
def json_parser(ch,failed_json):
  try:
    cleaned = clean_text(ch)
    return json.loads(ch)
  except json.JSONDecodeError:
        try:
            repaired = repair_json(ch)
            return json.loads(repaired)
        except Exception as e:
            return failed_json

In [39]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
#?????????????????????????????????????????????????????????????????

In [40]:
def extract_info(query,llm_chain,failed_json):
  try:
    raw_results=llm_chain.invoke(query)
    js=json_parser(raw_results,failed_json)
    if js == {}:
      return failed_json
    return js
  except Exception as e:
    logger.error(f"LLM failed: {e}.")
    return failed_json

In [41]:
def extract_cv_with_fallback(query, primary_chain, fallback_chain,fallback_chain2,failedjson):
    result = extract_info(query, primary_chain, failed_json)
    if result != failed_json and result !={}:
        logger.info("Primary LLM succeeded.")
        return result

    logger.warning("Primary LLM failed. Trying fallback LLM...")
    if fallback_chain:
        result = extract_info(query, fallback_chain, failed_json)
        if result != failed_json and result!={}:
            logger.info("Fallback LLM succeeded.")
            return result
        logger.warning("Fallback LLM also failed.")
    if fallback_chain2:
        result = extract_info(query, fallback_chain2, failed_json)
        if result != failed_json and result!={}:
            logger.info("Fallback LLM 2 succeeded.")
            return result
        logger.warning("Fallback LLM 2 also failed.")
    return failed_json


In [42]:
cv_failed_json = {
    "personal_information": {"full_name": None, "email": None, "phone": None},
    "education":[],
    "professional_summary": None,
    "work_experience": [],
    "certifications": [],
    "awards_and_achievements": [],
    "projects": [],
    "skills_and_interests": {
        "technical_skills": [],
        "soft_skills": [],
        "languages": [],
        "hobbies_and_interests": []
    },
    "volunteering": [],
    "publications": [],
    "website_and_social_links": {
        "linkedin": None,
        "github": None,
        "portfolio": None
    },
    "full_cv_text": None
}


personal information

In [43]:
query = """
You are an expert extraction algorithm.
You need to extract information from a candidate's cv
If you do not know the value of an attribute asked to extract
return null for the attribute's value.
Focus on these fields: personal_information .
Return it strictly in JSON format.in this form:
{
    "personal_information":
    {
    "full_name": "",
    "email": "",
    "phone": "",
     }

}

 If you cannot find a field, leave it empty.

Document content may have messy characters, special symbols, or concatenated text.
"""
failed_json=cv_failed_json['personal_information']
personal_information = extract_cv_with_fallback(query,llm_model,llm_model2,llm_model3,failed_json)
print(personal_information)

{'personal_information': {'full_name': 'Juan Jose Carin', 'email': 'juanjose.carin@gmail.com', 'phone': '650-336-4590'}}


links and websites

In [44]:
query_website_and_social_links = """
You are an expert extraction algorithm.
You need to extract information from a candidate's cv
If you do not know the value of an attribute asked to extract
return null for the attribute's value.
Focus on these fields: website_and_social_links .
Return it strictly in valid JSON format.in this form:
{
    "website_and_social_links":
    {
    "linkedin": "",
    "github": "",
    "portfolio": ""
     }

}

If you cannot find a field, leave it None.

Document content may have messy characters, special symbols, or concatenated text.
"""
failed_json=cv_failed_json['website_and_social_links']
website_and_social_links = extract_cv_with_fallback(query_website_and_social_links,llm_model,llm_model2,llm_model3,failed_json)
print(website_and_social_links)

{'website_and_social_links': {'linkedin': 'https://linkedin.com/in/juanjosecarin', 'github': None, 'portfolio': 'https://juanjocarin.github.io'}}


professional summary

In [45]:
query_professional_summary = """
You are an expert extraction algorithm.
You need to extract information from a candidate's cv
If you do not know the value of an attribute asked to extract
return null for the attribute's value.
Focus on these fields: professional_summary .
Return it strictly in JSON format.in this form:professional_summary = ""
} If you cannot find a field, leave it empty.

Document content may have messy characters, special symbols, or concatenated text.
"""
failed_json=cv_failed_json['professional_summary']
professional_summary = extract_cv_with_fallback(query_professional_summary,llm_model,llm_model2,llm_model3,failed_json)
print(professional_summary)

{'professional_summary': 'Passionate about data analysis and experiments, mainly focused on user behavior, experience, and engagement, with a solid background in data science and statistics, and extensive experience using data insights to drive business growth.'}


work experience

In [46]:
query_work_experience = """
You are an expert extraction algorithm.
You need to extract information from a candidate's cv
If you do not know the value of an attribute asked to extract
return null for the attribute's value.
Focus on these fields: work experience .
Return it strictly in JSON format.in this form:
{
    "work_experience": [
    {
        "job_title": "",
        "company": "",
        "location": "",
        "start_date": "",
        "end_date": "",
        "responsibilities": [],
        "achievements": []
    }
    ]
}


 If you cannot find a field, leave it empty.

Document content may have messy characters, special symbols, or concatenated text.
"""
failed_json=cv_failed_json['work_experience']
work_experience = extract_cv_with_fallback(query_work_experience,llm_model,llm_model2,llm_model3,failed_json)
print(work_experience)

{'work_experience': [{'job_title': 'Data Scientist', 'company': 'CONENTO', 'location': 'Madrid, Spain (working remotely)', 'start_date': 'Jan 2016', 'end_date': 'Mar 2016', 'responsibilities': ['Designed and implemented the ETL pipeline for a predictive model of traffic on the main roads in eastern Spain (a project for the Spanish government).', 'Automated scripts in R to extract, transform, clean (incl. anomaly detection), and load into MySQL data from multiple data sources: road traffic sensors, accidents, road works, weather.'], 'achievements': []}, {'job_title': 'Data Scientist', 'company': 'CONENTO', 'location': 'Madrid, Spain', 'start_date': 'Jun 2014', 'end_date': 'Sep 2014', 'responsibilities': ["Designed an experiment for Google Spain (conducted in October 2014) to measure the impact of YouTube ads on the sales of a car manufacturer's dealer network.", 'A matched-pair, cluster-randomized design, which involved selecting the test and control groups from a sample of 50+ cities i

Education

In [47]:
query_education = """
You are an expert extraction algorithm.
You need to extract information from a candidate's cv
If you do not know the value of an attribute asked to extract
return null for the attribute's value.
Focus on these fields: education .
Return it strictly in valid JSON format.in this form:
{
    "education": [
    {
        "degree": "",
        "field_of_study": "",
        "school": "",
        "location": "",
        "start_year": "",
        "end_year": "",
        "gpa": None
    }
    ]
}

 If you cannot find a field, leave it empty.

Document content may have messy characters, special symbols, or concatenated text.
"""

failed_json=cv_failed_json['education']
education = extract_cv_with_fallback(query_education,llm_model,llm_model2,llm_model3,failed_json)
print(education)

{'education': [{'degree': 'Master of Information and Data Science', 'field_of_study': '', 'school': 'University of California, Berkeley', 'location': 'Berkeley, CA', 'start_year': '', 'end_year': '2016', 'gpa': 3.93}, {'degree': 'M.S.', 'field_of_study': 'Statistical and Computational Information Processing', 'school': 'Universidad Politécnica de Madrid', 'location': 'Madrid, Spain', 'start_year': '', 'end_year': '2014', 'gpa': 3.69}, {'degree': 'M.S.', 'field_of_study': 'Telecommunication Engineering', 'school': 'Universidad Politécnica de Madrid', 'location': 'Madrid, Spain', 'start_year': '', 'end_year': '2005', 'gpa': 3.03}]}


certification

In [48]:
query_certification = """
You are an expert extraction algorithm.
Extract certifications from the CV.
If the CV does not contain certifications, return exactly this JSON:

{"certifications": []}

Otherwise, return:

{"certifications": [
    {"name": "", "issuer": "", "issue_date": ""}
]}

Do not invent or guess data.
"""
failed_json=cv_failed_json['certifications']
certification = extract_cv_with_fallback(query_certification,llm_model,llm_model2,llm_model3,failed_json)
print(certification)

{'certifications': []}


awards and achievements



In [49]:
query_awards_and_achievements = """
You are an expert information extraction algorithm.
Extract the candidate's awards and achievements from their CV.

Return strictly one JSON object with this structure:
{
    "awards_and_achievements": [
        {
            "title": "",
            "date": ""
        }
    ]
}

If there are no awards or achievements, return exactly:
{"awards_and_achievements": []}

- Do NOT include explanations, reasoning, or any extra text outside the JSON.
"""
failed_json=cv_failed_json['awards_and_achievements']
awards_and_achievements = extract_cv_with_fallback(query_awards_and_achievements,llm_model,llm_model2,llm_model3,failed_json)
print(awards_and_achievements)

{'awards_and_achievements': [{'title': 'Highest-rated professor in student surveys (4 of 6 terms)', 'date': 'Jul. 2002 – Jun. 2004'}, {'title': 'Promoted to head of sales after 5 months', 'date': 'Apr. 2008 – Jan. 2009'}, {'title': 'Exceeded sales target every year (2005‑2007) and achieved 60% of target in first 3 months of 2008', 'date': 'Sep. 2004 – Mar. 2008'}, {'title': 'Increased revenue by 6.3%, gross profit by 4.2%, operating income by 146%, and achieved 30% new customer ratio (3x growth)', 'date': ''}]}


projects

In [50]:
query_projects = """
You are an expert extraction algorithm.
You need to extract information from a candidate's cv
If you do not know the value of an attribute asked to extract
return null for the attribute's value.
Focus on these fields: projects .
Return it strictly in valid JSON format.in this form:
{
    'projects': [
        {
            "name": "",
            "description": "",
            "link": ""
        }
    ]
}

If you cannot find a field, leave it empty.

Document content may have messy characters, special symbols, or concatenated text.
"""
failed_json=cv_failed_json['projects']
projects = extract_cv_with_fallback(query_projects,llm_model,llm_model2,llm_model3,failed_json)
print(projects)

{'projects': [{'name': 'SmartCam', 'description': 'A scalable cloud‑based video monitoring system that features motion detection, face counting, and image recognition. Implemented with Python, OpenCV, TensorFlow, and AWS (EC2, S3, DynamoDB).', 'link': ''}, {'name': 'Shortest Path and PageRank on Wikipedia Graph', 'description': 'Implementation of the Shortest Path and PageRank algorithms using the Wikipedia graph dataset (≈\u202f500,000 nodes). Built with Hadoop, MrJob, Python, and AWS EC2/S3.', 'link': ''}, {'name': 'Forest Cover Type Prediction', 'description': 'Kaggle competition project predicting predominant tree cover type from cartographic variables (elevation, soil type) using random forests, SVMs, k‑NN, Naive Bayes, Gradient Descent, GMMs, etc.', 'link': ''}, {'name': 'Redefining the Job Search Process', 'description': 'Data pipeline that combines data from the Indeed API and the U.S. Census Bureau to select optimal locations for data scientists based on job postings, housing 

skills and interests

In [51]:
query_skills_and_interests = """
You are an expert extraction algorithm.
Extract only from the "Skills" section of a CV.
Focus strictly on these fields: technical_skills, soft_skills, languages, hobbies_and_interests.
Do not extract information from any other sections.
In the "languages" field:
- Include **only spoken or written languages** explicitly mentioned in the resume.
- Do not include programming or computer languages.
- If no spoken languages are present, return null.

Return strictly in JSON format, like this:
{
  "skills_and_interests": {
    "technical_skills": [],
    "soft_skills": [],
    "languages": [
      {
        "name": "",
        "proficiency": ""
      }
    ],
    "hobbies_and_interests": []
  }
}
Document content may have messy characters, special symbols, or concatenated text.


"""
failed_json = cv_failed_json['skills_and_interests']
skills_and_interests = extract_cv_with_fallback(query_skills_and_interests,llm_model,llm_model2,llm_model3,failed_json)
print(skills_and_interests)

{'skills_and_interests': {'technical_skills': ['R', 'Python', 'SQL', 'Hadoop', 'Hive', 'MrJob', 'Tableau', 'Git', 'AWS', 'SPSS', 'SAS', 'Matlab', 'Spark', 'Storm', 'Bash', 'EViews', 'Demetra+', 'D3.js', 'Gephi', 'Neo4j', 'QGIS'], 'soft_skills': [], 'languages': None, 'hobbies_and_interests': []}}


volunteering

In [52]:
query_volunteering = """
You are an expert extraction algorithm.
You need to extract information from a candidate's cv
If you do not know the value of an attribute asked to extract
return null for the attribute's value.
Focus on these fields: volunteering.
Return it strictly in valid JSON format.in this form:
{
    "volunteering": [
        {
            "organization": "",
            "role": "",
            "start_date": "",
            "end_date": "",
            "description": ""
        }
    ]
}

If you cannot find a field, leave it empty.

Document content may have messy characters, special symbols, or concatenated text.
"""
failed_json = cv_failed_json['volunteering']
volunteering = extract_cv_with_fallback(query_volunteering,llm_model,llm_model2,llm_model3,failed_json)
print(volunteering)

{'volunteering': []}


publications

In [53]:
query_publications = """
You are an expert extraction algorithm.
You need to extract information from a candidate's cv
If you do not know the value of an attribute asked to extract
return null for the attribute's value.
Focus on these fields: publications.
Return it strictly in valid JSON format.in this form:
{
    "publications": [
        {
            "title": "",
            "journal_or_conference": "",
            "publication_date": "",
            "url": "",
            "description": ""
        }
    ]
}
If you cannot find a field, leave it empty.

Document content may have messy characters, special symbols, or concatenated text.

"""
failed_json = cv_failed_json['publications']
publications = extract_cv_with_fallback(query_publications,llm_model,llm_model2,llm_model3,failed_json)
print(publications)

{'publications': []}


### extract information from results

In [54]:
print(personal_information)
print(education)
print(website_and_social_links)
print(professional_summary)
print(work_experience)
print(certification)
print(awards_and_achievements)
print(projects)
print(skills_and_interests)
print(volunteering)
print(publications)

{'personal_information': {'full_name': 'Juan Jose Carin', 'email': 'juanjose.carin@gmail.com', 'phone': '650-336-4590'}}
{'education': [{'degree': 'Master of Information and Data Science', 'field_of_study': '', 'school': 'University of California, Berkeley', 'location': 'Berkeley, CA', 'start_year': '', 'end_year': '2016', 'gpa': 3.93}, {'degree': 'M.S.', 'field_of_study': 'Statistical and Computational Information Processing', 'school': 'Universidad Politécnica de Madrid', 'location': 'Madrid, Spain', 'start_year': '', 'end_year': '2014', 'gpa': 3.69}, {'degree': 'M.S.', 'field_of_study': 'Telecommunication Engineering', 'school': 'Universidad Politécnica de Madrid', 'location': 'Madrid, Spain', 'start_year': '', 'end_year': '2005', 'gpa': 3.03}]}
{'website_and_social_links': {'linkedin': 'https://linkedin.com/in/juanjosecarin', 'github': None, 'portfolio': 'https://juanjocarin.github.io'}}
{'professional_summary': 'Passionate about data analysis and experiments, mainly focused on use

In [55]:
parsed_data = parse_with_rapidapi(DATA_PATH, API_KEY, API_HOST, API_URL)
parsed_data

{'success': True,
 'metadata': {'fileName': '/content/resume_juanjosecarin.pdf',
  'fileSize': 105901,
  'fileType': 'pdf',
  'processedAt': '2026-04-14T18:42:05.921Z'},
 'data': {'name': 'Juan Jose Carin',
  'email': 'juanjose.carin@gmail.com',
  'phone': '650-336-4590',
  'education': [{'degree': 'Master of Information and Data Science',
    'institution': 'University of California, Berkeley',
    'year': '2016'},
   {'degree': 'M.S. in Statistical and Computational Information Processing',
    'institution': 'Universidad Politécnica de Madrid',
    'year': '2014'},
   {'degree': 'M.S. in Telecommunication Engineering',
    'institution': 'Universidad Politécnica de Madrid',
    'year': '2005'}],
  'experience': [{'title': 'Data Scientist',
    'company': 'CONENTO',
    'duration': 'Jan. 2016- Mar. 2016',
    'description': 'Designed and implemented the ETL pipeline for a predictive model of traffic on the main roads in eastern Spain (a project for the Spanish government).\nAutomated

In [56]:
def safe_extract_section(section_name, section_data, default_value):
    """
    Safely extract the main content from a LLM output dictionary.
    Handles nested or flat dicts.
    """
    if isinstance(section_data, dict):
        if section_name in section_data:
            return section_data[section_name]
        return section_data

    # If not a dict, return default
    logger.warning(f"Section '{section_name}' missing or invalid. Using default value.")
    return default_value


In [57]:
def build_cv(sections):
    """
    Build a structured CV dictionary from multiple section outputs.
    sections: dict with keys like 'personal_information', 'education', etc.
    """
    defaults = {
        "personal_information": {"full_name": None, "email": None, "phone": None},
        "education": [],
        "website_and_social_links": {},
        "professional_summary": None,
        "work_experience": [],
        "certifications": [],
        "awards_and_achievements": [],
        "projects": [],
        "skills_and_interests": {
            "technical_skills": [],
            "soft_skills": [],
            "languages": [],
            "hobbies_and_interests": []
        },
        "volunteering": [],
        "publications": []
    }

    cv = {}
    for key, default in defaults.items():
        cv[key] = safe_extract_section(key, sections.get(key, {}), default)

    return cv


In [58]:
sections = {
    "personal_information": personal_information,
    "education": education,
    "website_and_social_links": website_and_social_links,
    "professional_summary": professional_summary,
    "work_experience": work_experience,
    "certifications": certification,
    "awards_and_achievements": awards_and_achievements,
    "projects": projects,
    "skills_and_interests": skills_and_interests,
    "volunteering": volunteering,
    "publications": publications
}

cv = build_cv(sections)
logger.info("CV built successfully")

In [59]:
cv

{'personal_information': {'full_name': 'Juan Jose Carin',
  'email': 'juanjose.carin@gmail.com',
  'phone': '650-336-4590'},
 'education': [{'degree': 'Master of Information and Data Science',
   'field_of_study': '',
   'school': 'University of California, Berkeley',
   'location': 'Berkeley, CA',
   'start_year': '',
   'end_year': '2016',
   'gpa': 3.93},
  {'degree': 'M.S.',
   'field_of_study': 'Statistical and Computational Information Processing',
   'school': 'Universidad Politécnica de Madrid',
   'location': 'Madrid, Spain',
   'start_year': '',
   'end_year': '2014',
   'gpa': 3.69},
  {'degree': 'M.S.',
   'field_of_study': 'Telecommunication Engineering',
   'school': 'Universidad Politécnica de Madrid',
   'location': 'Madrid, Spain',
   'start_year': '',
   'end_year': '2005',
   'gpa': 3.03}],
 'website_and_social_links': {'linkedin': 'https://linkedin.com/in/juanjosecarin',
  'github': None,
  'portfolio': 'https://juanjocarin.github.io'},
 'professional_summary': 'Pas

In [60]:
def verification(parsed_data,cv,failed_json):
  if cv["personal_information"] == failed_json['personal_information'] and parsed_data:
      cv["personal_information"] ['full_name']= parsed_data["data"]['name']
      cv["personal_information"] ['email']= parsed_data["data"]['email']
      cv['personal_information'] ['phone']= parsed_data["data"]['phone']
  if cv['education'] == failed_json['education']:
    for edu in parsed_data["data"]["education"]:
        cv['education'].append({
            'degree': edu.get('degree', None),
            'field_of_study': edu.get('field_of_study', None),
            'school': edu.get('institution', None),
            'location': edu.get('location', None),
            'start_year': edu.get('start_year', None),
            'end_year': edu.get('end_year', None),
            'gpa': edu.get('gpa', None)
        })
  if cv['work_experience'] == failed_json['work_experience']:
    for exp in parsed_data["data"]['experience']:
        cv['work_experience'].append({
            'job_title': exp.get('title', None),
            'company': exp.get('company', None),
            'location': exp.get('location', None),
            'start_date': exp.get('start_date', None),
            'end_date': exp.get('end_date', None),
            'responsibilities': exp.get('description', None),
            'achievements': exp.get('achievements', [])
        })

  if cv['skills_and_interests'] == failed_json['skills_and_interests']:
   cv['skills_and_interests']['technical_skills']=parsed_data["data"]['skills']
  return cv

In [61]:
cv=verification(parsed_data,cv,cv_failed_json)

In [62]:
cv['pages'] = len(raw_documents) if isinstance(raw_documents, list) else 0

In [63]:
cv["full_cv_text"]=clean_text(raw_documents[0].page_content)

In [64]:
json_data = json.dumps(cv, indent=4, ensure_ascii=False)
logger.info("CV JSON data:\n%s", json_data)

In [65]:
cv

{'personal_information': {'full_name': 'Juan Jose Carin',
  'email': 'juanjose.carin@gmail.com',
  'phone': '650-336-4590'},
 'education': [{'degree': 'Master of Information and Data Science',
   'field_of_study': '',
   'school': 'University of California, Berkeley',
   'location': 'Berkeley, CA',
   'start_year': '',
   'end_year': '2016',
   'gpa': 3.93},
  {'degree': 'M.S.',
   'field_of_study': 'Statistical and Computational Information Processing',
   'school': 'Universidad Politécnica de Madrid',
   'location': 'Madrid, Spain',
   'start_year': '',
   'end_year': '2014',
   'gpa': 3.69},
  {'degree': 'M.S.',
   'field_of_study': 'Telecommunication Engineering',
   'school': 'Universidad Politécnica de Madrid',
   'location': 'Madrid, Spain',
   'start_year': '',
   'end_year': '2005',
   'gpa': 3.03}],
 'website_and_social_links': {'linkedin': 'https://linkedin.com/in/juanjosecarin',
  'github': None,
  'portfolio': 'https://juanjocarin.github.io'},
 'professional_summary': 'Pas

In [66]:
file_path = f"/content/{cv['personal_information']['full_name']}_cv.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(cv, f, indent=4, ensure_ascii=False)

print(f"✅ JSON saved successfully at: {file_path}")

✅ JSON saved successfully at: /content/Juan Jose Carin_cv.json


In [67]:
def clean_vector_store(vector_store):
    try:
        result = vector_store.get()
        all_ids = result.get("ids", []) if result else []
        if all_ids:
            vector_store.delete(ids=all_ids)
            logger.info("🧹 Deleted %d embeddings from 'cv_collection'.", len(all_ids))
        else:
            logger.info("✅ Collection already empty.")
    except Exception as e:
        logger.error("⚠️ Could not clean Chroma DB: %s", e)

In [68]:
clean_vector_store(vector_store)

In [69]:
"""personal_information = {
    "full_name": "",
    "email": "",
    "phone": "",
}

website_and_social_links = {
    "linkedin": "",
    "github": "",
    "portfolio": ""
}

professional_summary = ""

work_experience = [
    {
        "job_title": "",
        "company": "",
        "location": "",
        "start_date": "",
        "end_date": "",
        "responsibilities": [],
        "achievements": []
    }
]

education = [
    {
        "degree": "",
        "field_of_study": "",
        "school": "",
        "location": "",
        "start_year": "",
        "end_year": "",
        "gpa": None
    }
]

certifications = [
    {
        "name": "",
        "issuer": "",
        "issue_date": ""
    }
]

awards_and_achievements = [
    {
        "title": "",
        "date": ""
    }
]

projects = [
    {
        "name": "",
        "description": "",
        "link": ""
    }
]

skills_and_interests = {
    "technical_skills": [],
    "soft_skills": [],
    "languages": [{
      "name": "",
      "proficiency": ""
    }],
    "hobbies_and_interests": []
}

volunteering = [
    {
        "organization": "",
        "role": "",
        "start_date": "",
        "end_date": "",
        "description": ""
    }
]

publications = [
    {
        "title": "",
        "journal_or_conference": "",
        "publication_date": "",
        "url": "",
        "description": ""
    }
]
pages=None
score = None

feedback = {
    "weaknesses": "",
    "strengths": "",
    "suggestions": ""
}
"""


'personal_information = {\n    "full_name": "",\n    "email": "",\n    "phone": "",\n}\n\nwebsite_and_social_links = {\n    "linkedin": "",\n    "github": "",\n    "portfolio": ""\n}\n\nprofessional_summary = ""\n\nwork_experience = [\n    {\n        "job_title": "",\n        "company": "",\n        "location": "",\n        "start_date": "",\n        "end_date": "",\n        "responsibilities": [],\n        "achievements": []\n    }\n]\n\neducation = [\n    {\n        "degree": "",\n        "field_of_study": "",\n        "school": "",\n        "location": "",\n        "start_year": "",\n        "end_year": "",\n        "gpa": None\n    }\n]\n\ncertifications = [\n    {\n        "name": "",\n        "issuer": "",\n        "issue_date": ""\n    }\n]\n\nawards_and_achievements = [\n    {\n        "title": "",\n        "date": ""\n    }\n]\n\nprojects = [\n    {\n        "name": "",\n        "description": "",\n        "link": ""\n    }\n]\n\nskills_and_interests = {\n    "technical_skills